# Phase 2 — Classical ML
## Day 6: Train/Test Splits & Sklearn Pipelines
**Date:** 2026-04-23

### What you'll learn today
- How to split data into train and test sets (and why)
- K-Fold and Stratified K-Fold cross-validation
- Using `cross_val_score` for quick model evaluation
- Building sklearn `Pipeline` for clean, reproducible workflows
- Using `ColumnTransformer` to handle mixed column types

In [ ]:
# Setup — install and import everything we need
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

print("All imports loaded!")

In [ ]:
# Sample data — a fake "loan approval" dataset
np.random.seed(42)
n = 200

df = pd.DataFrame({
    "age": np.random.randint(20, 65, n),
    "income": np.random.normal(50000, 15000, n).round(2),
    "credit_score": np.random.randint(300, 850, n),
    "employment": np.random.choice(["salaried", "self-employed", "freelance"], n),
    "education": np.random.choice(["high_school", "bachelors", "masters", "phd"], n),
})

# Target: approved (1) or not (0), loosely based on credit score + income
prob = 1 / (1 + np.exp(-(df["credit_score"] - 550) / 100 - (df["income"] - 40000) / 30000))
df["approved"] = (np.random.random(n) < prob).astype(int)

# Sprinkle in some missing values
df.loc[np.random.choice(n, 10, replace=False), "income"] = np.nan
df.loc[np.random.choice(n, 8, replace=False), "credit_score"] = np.nan

print(f"Shape: {df.shape}")
print(f"Target balance:\n{df['approved'].value_counts()}")
df.head()

## 1. Train/Test Split

The most basic idea in ML evaluation: you can't test your model on data it already saw during training. That's like giving a student the exam answers, then being impressed they scored 100%.

`train_test_split` randomly splits your data into two parts. The model learns from the training set and gets evaluated on the test set. A typical split is 80/20 or 75/25. Always set `random_state` so your results are reproducible.

In [ ]:
# Separate features (X) and target (y)
X = df.drop("approved", axis=1)
y = df["approved"]

# Basic train/test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set:     {X_test.shape[0]} rows")
print(f"\nTarget distribution in train:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTarget distribution in test:\n{y_test.value_counts(normalize=True).round(3)}")

In [ ]:
# Using stratify to keep target proportions equal in both sets
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("With stratify=y:")
print(f"Train target dist:\n{y_train_s.value_counts(normalize=True).round(3)}")
print(f"\nTest target dist:\n{y_test_s.value_counts(normalize=True).round(3)}")
print("\nNotice how both sets now have nearly identical class proportions!")

## 2. K-Fold Cross-Validation

A single train/test split can be misleading. Maybe you got lucky (or unlucky) with which rows ended up where. K-Fold fixes this by splitting the data into K equal parts ("folds"), then training K times. Each time, one fold is the test set and the rest are training.

This gives you K accuracy scores instead of one. You can look at the mean and standard deviation to understand how stable your model really is. The standard choice is K=5 or K=10.

In [ ]:
# KFold — see how it splits indices
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("KFold splits (showing first 5 indices of each):\n")
for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    print(f"Fold {fold}: train={train_idx[:5]}...  test={test_idx[:5]}...")
    print(f"         train size={len(train_idx)}, test size={len(test_idx)}")

### StratifiedKFold

Regular KFold doesn't care about your target distribution. If your dataset is 90% class 0 and 10% class 1, one fold might end up with 0% of class 1. StratifiedKFold ensures each fold has roughly the same class proportions as the full dataset. **Always use StratifiedKFold for classification.**

In [ ]:
# Compare KFold vs StratifiedKFold target distribution per fold
print("=== Regular KFold ===")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, test_idx) in enumerate(kf.split(X, y), 1):
    pct = y.iloc[test_idx].mean()
    print(f"  Fold {fold}: % approved in test = {pct:.3f}")

print(f"\n  Overall % approved = {y.mean():.3f}")

print("\n=== StratifiedKFold ===")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    pct = y.iloc[test_idx].mean()
    print(f"  Fold {fold}: % approved in test = {pct:.3f}")

print(f"\n  Overall % approved = {y.mean():.3f}")
print("\nStratified folds are much closer to the overall distribution!")

## 3. cross_val_score

Doing K-Fold manually (looping, fitting, scoring) works, but sklearn gives you a one-liner: `cross_val_score`. It handles the splitting, training, and scoring for you. You just pass a model, features, target, and number of folds.

By default it uses the model's `.score()` method (accuracy for classifiers). You can change the metric with the `scoring` parameter.

In [ ]:
# cross_val_score needs numeric-only features, so let's use just the numeric cols for now
numeric_cols = ["age", "income", "credit_score"]
X_numeric = df[numeric_cols].fillna(df[numeric_cols].median())

# Quick cross-validation with a Decision Tree
dt = DecisionTreeClassifier(max_depth=3, random_state=42)

scores = cross_val_score(dt, X_numeric, y, cv=5, scoring="accuracy")
print(f"Fold scores: {scores.round(3)}")
print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

# Try with StratifiedKFold explicitly
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_strat = cross_val_score(dt, X_numeric, y, cv=skf, scoring="accuracy")
print(f"\nStratified fold scores: {scores_strat.round(3)}")
print(f"Mean accuracy: {scores_strat.mean():.3f} (+/- {scores_strat.std():.3f})")

In [ ]:
# Common scoring options you should know
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    scores = cross_val_score(dt, X_numeric, y, cv=5, scoring=metric)
    print(f"{metric:>12}: {scores.mean():.3f} (+/- {scores.std():.3f})")

## 4. Sklearn Pipeline

A Pipeline chains preprocessing and modeling steps into a single object. Instead of manually scaling, encoding, imputing, then fitting, you define the steps once and call `.fit()` / `.predict()` on the pipeline.

Why bother? Three big reasons:
1. **No data leakage.** The pipeline ensures transformations are fit only on training data.
2. **Cleaner code.** One object does everything.
3. **Works with cross_val_score.** You can cross-validate the entire pipeline, preprocessing included.

In [ ]:
# A simple pipeline: impute missing values -> scale -> classify
simple_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=42))
])

# Fit on training data, predict on test data
simple_pipe.fit(X_train[numeric_cols], y_train)
y_pred = simple_pipe.predict(X_test[numeric_cols])
print(f"Pipeline accuracy: {accuracy_score(y_test, y_pred):.3f}")

# You can access individual steps
print(f"\nPipeline steps: {[name for name, _ in simple_pipe.steps]}")
print(f"Scaler mean: {simple_pipe.named_steps['scaler'].mean_.round(2)}")

In [ ]:
# Cross-validate the entire pipeline — this is the clean way to do it
scores = cross_val_score(simple_pipe, X_train[numeric_cols], y_train, cv=5, scoring="accuracy")
print(f"Pipeline cross-val scores: {scores.round(3)}")
print(f"Mean: {scores.mean():.3f} (+/- {scores.std():.3f})")
print("\nThe pipeline handles imputation + scaling + fitting inside each fold!")

## 5. ColumnTransformer

Real datasets have mixed types: numbers and categories. You need different preprocessing for each. `ColumnTransformer` lets you apply different pipelines to different columns, then combines the results.

Think of it as a router: "send these columns through pipeline A, those columns through pipeline B, then stitch the outputs together."

In [ ]:
# Define column groups
num_features = ["age", "income", "credit_score"]
cat_features = ["employment", "education"]

# Numeric pipeline: impute missing with median, then scale
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline: impute missing with most frequent, then one-hot encode
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# ColumnTransformer combines them
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

# See what the output looks like
X_transformed = preprocessor.fit_transform(X_train)
print(f"Original shape: {X_train.shape}")
print(f"Transformed shape: {X_transformed.shape}")
print(f"\nNew columns: {num_features + list(preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(cat_features))}")

In [ ]:
# The full pipeline: ColumnTransformer + classifier
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000))
])

# Now we can use ALL columns, including categorical ones
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)
print(f"Full pipeline accuracy: {accuracy_score(y_test, y_pred):.3f}")

# Cross-validate the whole thing
scores = cross_val_score(full_pipeline, X, y, cv=5, scoring="accuracy")
print(f"\nCross-val scores: {scores.round(3)}")
print(f"Mean: {scores.mean():.3f} (+/- {scores.std():.3f})")

In [ ]:
# Swap the classifier easily — just change one step
full_pipeline_dt = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=4, random_state=42))
])

scores_dt = cross_val_score(full_pipeline_dt, X, y, cv=5, scoring="accuracy")
print(f"Decision Tree:       {scores_dt.mean():.3f} (+/- {scores_dt.std():.3f})")

scores_lr = cross_val_score(full_pipeline, X, y, cv=5, scoring="accuracy")
print(f"Logistic Regression: {scores_lr.mean():.3f} (+/- {scores_lr.std():.3f})")
print("\nSame preprocessing, different models. That's the power of Pipelines.")

## Tricky Bits

These are common mistakes that trip up beginners. Run the code below to see what goes wrong and why.

In [ ]:
# MISTAKE 1: Data leakage — scaling BEFORE splitting
# This is WRONG because the scaler sees test data statistics

# Wrong way:
scaler_wrong = StandardScaler()
X_all_scaled = scaler_wrong.fit_transform(X_numeric)  # Fit on ALL data!
X_tr_wrong, X_te_wrong, y_tr, y_te = train_test_split(X_all_scaled, y, test_size=0.2, random_state=42)

# Right way:
X_tr_right, X_te_right, y_tr, y_te = train_test_split(X_numeric, y, test_size=0.2, random_state=42)
scaler_right = StandardScaler()
X_tr_right = scaler_right.fit_transform(X_tr_right)   # Fit only on train
X_te_right = scaler_right.transform(X_te_right)        # Transform test (no fit!)

print("Wrong way: scaler saw ALL data including test set = DATA LEAKAGE")
print("Right way: scaler fit only on training data")
print("\nPipelines prevent this automatically — the pipeline fits transforms only on training data")

In [ ]:
# MISTAKE 2: Forgetting to set shuffle=True in KFold
# If your data is sorted by target, you'll get terrible results

# Make a sorted dataset
y_sorted = pd.Series([0]*100 + [1]*100)
X_dummy = pd.DataFrame({"x": np.random.randn(200)})

# Without shuffle — disaster!
kf_no_shuffle = KFold(n_splits=5, shuffle=False)
for fold, (train_idx, test_idx) in enumerate(kf_no_shuffle.split(X_dummy, y_sorted), 1):
    pct = y_sorted.iloc[test_idx].mean()
    if fold <= 3:
        print(f"  Fold {fold} (no shuffle): % class 1 = {pct:.1f}")

print("  ... some folds have 0% or 100% of class 1!")
print("\n  Fix: use shuffle=True, or better yet, use StratifiedKFold")

In [ ]:
# MISTAKE 3: Using fit_transform on test data
try:
    scaler = StandardScaler()
    scaler.fit_transform(X_tr_right)
    # WRONG: fit_transform on test recalculates mean/std from test data
    X_te_leaked = scaler.fit_transform(X_te_right)  # BAD!
    print("This ran but it's WRONG — you just re-fit the scaler on test data")
    print("The test data will be scaled using its own mean/std, not train's")
    print("\nAlways use .transform() (not .fit_transform()) on test data!")
except Exception as e:
    print(f"Error: {e}")

## Trick Questions

Test your understanding. Try to answer before revealing the solution.

<details>
<summary><strong>Q1: You have 1000 samples and use 5-fold CV. How many times does each sample appear in a test set?</strong></summary>

Exactly once. Each fold uses a different 200 samples as the test set, and after 5 folds every sample has been tested exactly once.
</details>

<details>
<summary><strong>Q2: If your cross-val accuracy is 95% but your test set accuracy is 70%, what happened?</strong></summary>

You probably have data leakage. The most common cause: you preprocessed (scaled, imputed, encoded) the data before splitting, so the cross-validation folds "saw" information from their own test portions. Always put preprocessing inside the pipeline.
</details>

<details>
<summary><strong>Q3: Can you use cross_val_score with a Pipeline?</strong></summary>

Yes! That's one of the biggest benefits. The pipeline ensures that preprocessing is re-fit on each fold's training data, preventing leakage. Just pass the pipeline as the estimator argument.
</details>

<details>
<summary><strong>Q4: What does `remainder="passthrough"` do in ColumnTransformer?</strong></summary>

By default, any columns not listed in the ColumnTransformer are dropped. Setting `remainder="passthrough"` keeps them as-is in the output. Useful when you only want to transform some columns and leave the rest untouched.
</details>

<details>
<summary><strong>Q5: Why is StratifiedKFold better than KFold for classification?</strong></summary>

StratifiedKFold preserves the class distribution in every fold. If your dataset is 80/20, each fold will also be about 80/20. Regular KFold could give you a fold with no positive examples, leading to meaningless metrics.
</details>

## Exercises

Fill in the `___` blanks and run each cell. The `assert` statements will tell you if you got it right.

In [ ]:
# Exercise 1: Split data with 30% test size and stratification
X_tr1, X_te1, y_tr1, y_te1 = train_test_split(
    X, y, test_size=___, stratify=___, random_state=42
)

assert X_te1.shape[0] == 60, "Test set should have 60 rows (30% of 200)"
assert abs(y_tr1.mean() - y_te1.mean()) < 0.05, "Stratification should balance classes"
print("Exercise 1 passed!")

In [ ]:
# Exercise 2: Create a StratifiedKFold with 10 folds, shuffled
skf_ex = ___(n_splits=___, shuffle=___, random_state=42)

folds = list(skf_ex.split(X, y))
assert len(folds) == 10, "Should have 10 folds"
assert len(folds[0][1]) == 20, "Each test fold should have 20 samples (200/10)"
print("Exercise 2 passed!")

In [ ]:
# Exercise 3: Use cross_val_score with f1 scoring
dt_ex = DecisionTreeClassifier(max_depth=3, random_state=42)
f1_scores = cross_val_score(dt_ex, X_numeric, y, cv=5, scoring=___)

assert len(f1_scores) == 5, "Should have 5 scores"
assert all(0 <= s <= 1 for s in f1_scores), "F1 scores should be between 0 and 1"
print(f"F1 scores: {f1_scores.round(3)}")
print("Exercise 3 passed!")

In [ ]:
# Exercise 4: Build a pipeline with SimpleImputer (mean strategy) + StandardScaler + LogisticRegression
pipe_ex = Pipeline([
    ("imputer", SimpleImputer(strategy=___)),
    ("scaler", ___()),
    ("model", ___(random_state=42))
])

pipe_ex.fit(X_train[numeric_cols], y_train)
acc = pipe_ex.score(X_test[numeric_cols], y_test)
assert 0.5 < acc < 1.0, "Accuracy should be reasonable"
print(f"Pipeline accuracy: {acc:.3f}")
print("Exercise 4 passed!")

In [ ]:
# Exercise 5: Create a ColumnTransformer that scales numeric cols and one-hot encodes categorical cols
# Use the num_features and cat_features lists defined earlier

ct_ex = ColumnTransformer([
    ("numbers", ___, num_features),
    ("categories", ___, cat_features)
])

result = ct_ex.fit_transform(X_train)
assert result.shape[0] == X_train.shape[0], "Same number of rows"
assert result.shape[1] > X_train.shape[1], "Should have more columns after one-hot encoding"
print(f"Transformed shape: {result.shape}")
print("Exercise 5 passed!")

In [ ]:
# Exercise 6: Build a full pipeline (ColumnTransformer + model) and cross-validate it
# Use the preprocessor we defined earlier and a DecisionTreeClassifier

full_pipe_ex = Pipeline([
    (___,  preprocessor),
    (___, DecisionTreeClassifier(max_depth=4, random_state=42))
])

cv_scores = cross_val_score(full_pipe_ex, X, y, cv=5)
assert len(cv_scores) == 5
assert cv_scores.mean() > 0.5, "Should do better than random"
print(f"Full pipeline CV scores: {cv_scores.round(3)}")
print(f"Mean: {cv_scores.mean():.3f}")
print("Exercise 6 passed!")

In [ ]:
# Exercise 7: Create a ColumnTransformer with remainder="passthrough"
# Only scale the "income" column, pass everything else through unchanged

ct_pass = ColumnTransformer([
    ("scale_income", StandardScaler(), ___)
], remainder=___)

result_pass = ct_pass.fit_transform(X_train)
assert result_pass.shape[1] == X_train.shape[1], "All columns should be present"
print(f"Shape: {result_pass.shape}")
print("Exercise 7 passed!")

### Solutions

<details>
<summary><strong>Exercise 1</strong></summary>

```python
X_tr1, X_te1, y_tr1, y_te1 = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
```
</details>

<details>
<summary><strong>Exercise 2</strong></summary>

```python
skf_ex = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
```
</details>

<details>
<summary><strong>Exercise 3</strong></summary>

```python
f1_scores = cross_val_score(dt_ex, X_numeric, y, cv=5, scoring="f1")
```
</details>

<details>
<summary><strong>Exercise 4</strong></summary>

```python
pipe_ex = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])
```
</details>

<details>
<summary><strong>Exercise 5</strong></summary>

```python
ct_ex = ColumnTransformer([
    ("numbers", StandardScaler(), num_features),
    ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features)
])
```
</details>

<details>
<summary><strong>Exercise 6</strong></summary>

```python
full_pipe_ex = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=4, random_state=42))
])
```
</details>

<details>
<summary><strong>Exercise 7</strong></summary>

```python
ct_pass = ColumnTransformer([
    ("scale_income", StandardScaler(), ["income"])
], remainder="passthrough")
```
</details>

## Cumulative Review Exercises (Days 1-5)

Mixed exercises covering Pandas, NumPy, Data Cleaning, Faker, and PyTorch basics.

In [ ]:
# Review 1 (Day 1 - Pandas): Filter df to only rows where income > 50000
high_income = df[df[___] > ___]
assert high_income.shape[0] < df.shape[0], "Should have fewer rows"
assert all(high_income["income"].dropna() > 50000), "All incomes should be > 50000"
print(f"High income rows: {high_income.shape[0]}")
print("Review 1 passed!")

In [ ]:
# Review 2 (Day 1 - Pandas): Get the mean credit_score grouped by education level
grouped = df.groupby(___)[___].___()
assert len(grouped) == 4, "Should have 4 education levels"
print(grouped)
print("Review 2 passed!")

In [ ]:
# Review 3 (Day 2 - NumPy): Create a 3x3 array of ones and multiply by 5 using broadcasting
ones = np.___((___, ___))
result_np = ones * ___
assert result_np.shape == (3, 3)
assert np.all(result_np == 5)
print(result_np)
print("Review 3 passed!")

In [ ]:
# Review 4 (Day 2 - NumPy): Compute the mean along axis=0 (column-wise)
arr = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
col_means = np.mean(arr, axis=___)
assert col_means.shape == (3,)
assert list(col_means) == [4.0, 5.0, 6.0]
print(f"Column means: {col_means}")
print("Review 4 passed!")

In [ ]:
# Review 5 (Day 3 - Data Cleaning): Count missing values in each column of df
missing = df.___.___()
assert missing["income"] == 10, "Should have 10 missing income values"
print(missing)
print("Review 5 passed!")

In [ ]:
# Review 6 (Day 3 - Data Cleaning): Fill missing income values with the median
df_clean = df.copy()
median_income = df_clean["income"].___()
df_clean["income"] = df_clean["income"].___(median_income)
assert df_clean["income"].isna().sum() == 0, "No missing values after fill"
print(f"Median income: {median_income:.2f}")
print("Review 6 passed!")

In [ ]:
# Review 7 (Day 4 - Python Core): Write a list comprehension to get squares of even numbers from 0-9
squares = [x**2 for x in range(10) if x % ___ == ___]
assert squares == [0, 4, 16, 36, 64]
print(f"Squares of even numbers: {squares}")
print("Review 7 passed!")

In [ ]:
# Review 8 (Day 4 - Python Core): Use try/except to handle a ValueError
def safe_int(value):
    try:
        return ___(value)
    except ___:
        return None

assert safe_int("42") == 42
assert safe_int("hello") is None
assert safe_int("3.14") is None
print("Review 8 passed!")

In [ ]:
# Review 9 (Day 5 - PyTorch): Create a 2x3 tensor of zeros
import torch
t = torch.___((___, ___))
assert t.shape == (2, 3)
assert t.sum().item() == 0
print(t)
print("Review 9 passed!")

In [ ]:
# Review 10 (Day 5 - PyTorch): Convert a numpy array to a torch tensor and back
arr_r10 = np.array([1.0, 2.0, 3.0])
tensor_r10 = torch.___(arr_r10)
back_to_numpy = tensor_r10.___()
assert isinstance(tensor_r10, torch.Tensor)
assert isinstance(back_to_numpy, np.ndarray)
assert list(back_to_numpy) == [1.0, 2.0, 3.0]
print("Review 10 passed!")

### Cumulative Review Solutions

<details>
<summary><strong>Review 1</strong></summary>

```python
high_income = df[df["income"] > 50000]
```
</details>

<details>
<summary><strong>Review 2</strong></summary>

```python
grouped = df.groupby("education")["credit_score"].mean()
```
</details>

<details>
<summary><strong>Review 3</strong></summary>

```python
ones = np.ones((3, 3))
result_np = ones * 5
```
</details>

<details>
<summary><strong>Review 4</strong></summary>

```python
col_means = np.mean(arr, axis=0)
```
</details>

<details>
<summary><strong>Review 5</strong></summary>

```python
missing = df.isna().sum()
```
</details>

<details>
<summary><strong>Review 6</strong></summary>

```python
median_income = df_clean["income"].median()
df_clean["income"] = df_clean["income"].fillna(median_income)
```
</details>

<details>
<summary><strong>Review 7</strong></summary>

```python
squares = [x**2 for x in range(10) if x % 2 == 0]
```
</details>

<details>
<summary><strong>Review 8</strong></summary>

```python
def safe_int(value):
    try:
        return int(value)
    except ValueError:
        return None
```
</details>

<details>
<summary><strong>Review 9</strong></summary>

```python
t = torch.zeros((2, 3))
```
</details>

<details>
<summary><strong>Review 10</strong></summary>

```python
tensor_r10 = torch.from_numpy(arr_r10)
back_to_numpy = tensor_r10.numpy()
```
</details>

In [ ]:
# Cheat Sheet — Train/Test & Pipelines

cheat = """
╔══════════════════════════════════════════════════════════════════╗
║              TRAIN/TEST & PIPELINES CHEAT SHEET                 ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  TRAIN/TEST SPLIT                                                ║
║  X_tr, X_te, y_tr, y_te = train_test_split(                    ║
║      X, y, test_size=0.2, random_state=42, stratify=y)         ║
║                                                                  ║
║  K-FOLD CROSS-VALIDATION                                        ║
║  kf = KFold(n_splits=5, shuffle=True, random_state=42)         ║
║  skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)║
║  * Always use StratifiedKFold for classification                ║
║                                                                  ║
║  CROSS_VAL_SCORE                                                 ║
║  scores = cross_val_score(model, X, y, cv=5, scoring="f1")     ║
║  print(f"Mean: {scores.mean():.3f} +/- {scores.std():.3f}")    ║
║  Scoring: accuracy, precision, recall, f1, roc_auc             ║
║                                                                  ║
║  PIPELINE                                                        ║
║  pipe = Pipeline([                                               ║
║      ("imputer", SimpleImputer(strategy="median")),             ║
║      ("scaler", StandardScaler()),                              ║
║      ("model", LogisticRegression())                            ║
║  ])                                                              ║
║  pipe.fit(X_train, y_train)                                     ║
║  pipe.predict(X_test)                                           ║
║                                                                  ║
║  COLUMN TRANSFORMER                                              ║
║  ct = ColumnTransformer([                                        ║
║      ("num", num_pipeline, num_cols),                           ║
║      ("cat", cat_pipeline, cat_cols)                            ║
║  ], remainder="drop")   # or "passthrough"                     ║
║                                                                  ║
║  FULL PIPELINE                                                   ║
║  full = Pipeline([("preprocess", ct), ("model", clf)])          ║
║  cross_val_score(full, X, y, cv=5)  # no leakage!              ║
║                                                                  ║
║  KEY RULES                                                       ║
║  * Never fit_transform on test data — use .transform() only    ║
║  * Put all preprocessing inside the Pipeline                    ║
║  * Use stratify=y for classification splits                     ║
║  * Set random_state for reproducibility                         ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(cheat)

---

**Next up: Day 7 — LogRegAndTrees**

You'll dig into Logistic Regression (coefficients, regularization) and Decision Trees (gini vs entropy, max_depth, overfitting). Two of the most important classical ML models!